# Import libraries

In [1]:
import os
import glob
import json
import time
import gc
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

import lightgbm as lgb
import optuna

print("LightGBM version:", lgb.__version__)
print("Optuna version:", optuna.__version__)

LightGBM version: 4.6.0
Optuna version: 4.8.0


# Config

In [2]:
RANDOM_STATE = 42
INPUT_ROOT = "/opt/spark/workspace/input/data"
OUTPUT_DIR = "/opt/spark/workspace/output/lgbm_experiment"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# PICKUP_COL = "tpep_pickup_datetime"
# DROPOFF_COL = "tpep_dropoff_datetime"

TARGET_PROCESSING_DATE_HOUR = pd.Timestamp("2025-01-15 12:00:00")
PREVIOUS_BATCH_OFFSET = pd.Timedelta(hours=12)

HISTORY_OFFSETS = [
    pd.Timedelta(days=1),
    pd.Timedelta(days=7),
    pd.Timedelta(days=14),
]

# Sampling
MIN_ROWS_PER_GROUP = 100
TARGET_SAMPLE_PER_GROUP = 10_000

# Nếu True: mỗi processing_date_hour trong nhóm label = 0 lấy tối đa TARGET_SAMPLE_PER_GROUP dòng
# Nếu False: lấy toàn bộ rồi sample tổng
SAMPLE_EACH_COMPARISON_BATCH = True

# Train / valid / test split
TEST_SIZE = 0.15
VALID_SIZE = 0.15
VALID_SIZE_FROM_TRAIN = VALID_SIZE / (1 - TEST_SIZE)

# Optuna tuning
N_TRIALS = 100

# LightGBM
N_ESTIMATORS_MAX = 2000
EARLY_STOPPING_ROUNDS = 100

# Data Loading

In [3]:
parquet_files = glob.glob(f"{INPUT_ROOT}/**/*.parquet", recursive=True)
print("Found parquet files:")
for f in parquet_files:
    print(f)


Found parquet files:
/opt/spark/workspace/input/data/silver_yellow_tripdata_2025-01.parquet


In [4]:
DATA_PATH = "/opt/spark/workspace/input/data/silver_yellow_tripdata_2025-01.parquet"
print("Using:", DATA_PATH)

Using: /opt/spark/workspace/input/data/silver_yellow_tripdata_2025-01.parquet


In [5]:
df_raw = pd.read_parquet(DATA_PATH)
print(df_raw.shape)
display(df_raw.head())
print(df_raw.dtypes)

(3326978, 22)


,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,date_hour,processing_date_hour
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,...,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0,2025-01-01 00:00:00,2025-01-01 12:00:00
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,...,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0,2025-01-01 00:00:00,2025-01-01 12:00:00
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,...,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0,2025-01-01 01:00:00,2025-01-01 12:00:00
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,...,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0,2025-01-01 00:00:00,2025-01-01 12:00:00
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,...,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0,2025-01-01 00:00:00,2025-01-01 12:00:00


vendorid                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
ratecodeid                      float64
store_and_fwd_flag       string[python]
pulocationid                      int32
dolocationid                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
airport_fee                     float64
cbd_congestion_fee              float64
date_hour                datetime64[us]
processing_date_hour     datetime64[us]
dtype: object


# Data Pre-processing

In [8]:
print("Min tpep_pickup_datetime:", df_raw["tpep_pickup_datetime"].min())
print("Max tpep_pickup_datetime:", df_raw["tpep_pickup_datetime"].max())
print("Min tpep_dropoff_datetime :", df_raw["tpep_dropoff_datetime"].min())
print("Max tpep_dropoff_datetime :", df_raw["tpep_dropoff_datetime"].max())


Min tpep_pickup_datetime: 2024-12-31 23:25:38
Max tpep_pickup_datetime: 2025-01-31 23:44:00
Min tpep_dropoff_datetime : 2024-12-31 23:55:13
Max tpep_dropoff_datetime : 2025-01-31 23:44:59


In [7]:
print("After datetime parsing:", df_raw.shape)
print("Min date_hour :", df_raw["date_hour"].min())
print("Max date_hour :", df_raw["date_hour"].max())
print("Min processing_date_hour:", df_raw["processing_date_hour"].min())
print("Max processing_date_hour:", df_raw["processing_date_hour"].max())

After datetime parsing: (3326978, 22)
Min date_hour : 2025-01-01 00:00:00
Max date_hour : 2025-01-31 23:00:00
Min processing_date_hour: 2025-01-01 12:00:00
Max processing_date_hour: 2025-02-01 00:00:00


In [9]:
print(df_raw.shape)
display(df_raw.head())
print(df_raw.dtypes)

(3326978, 22)


,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,date_hour,processing_date_hour
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,...,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0,2025-01-01 00:00:00,2025-01-01 12:00:00
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,...,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0,2025-01-01 00:00:00,2025-01-01 12:00:00
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,...,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0,2025-01-01 01:00:00,2025-01-01 12:00:00
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,...,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0,2025-01-01 00:00:00,2025-01-01 12:00:00
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,...,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0,2025-01-01 00:00:00,2025-01-01 12:00:00


vendorid                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
ratecodeid                      float64
store_and_fwd_flag       string[python]
pulocationid                      int32
dolocationid                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
airport_fee                     float64
cbd_congestion_fee              float64
date_hour                datetime64[us]
processing_date_hour     datetime64[us]
dtype: object


In [10]:
processing_counts = (
    df_raw
    .groupby("processing_date_hour", as_index=False)
    .size()
    .rename(columns={"size": "row_count"})
    .sort_values("processing_date_hour")
    .reset_index(drop=True)
)

display(processing_counts.head())
display(processing_counts.tail())

,processing_date_hour,row_count
0,2025-01-01 12:00:00,36464
1,2025-01-02 00:00:00,45227
2,2025-01-02 12:00:00,19740
3,2025-01-03 00:00:00,61896
4,2025-01-03 12:00:00,21563


,processing_date_hour,row_count
57,2025-01-30 00:00:00,87625
58,2025-01-30 12:00:00,32929
59,2025-01-31 00:00:00,94242
60,2025-01-31 12:00:00,31944
61,2025-02-01 00:00:00,93156


In [11]:
target_processing_hour = pd.to_datetime(TARGET_PROCESSING_DATE_HOUR)
comparison_processing_hours = []
comparison_processing_hours.append(
    target_processing_hour - PREVIOUS_BATCH_OFFSET
)

comparison_processing_hours.extend([
    target_processing_hour - offset
    for offset in HISTORY_OFFSETS
])

target_count = int(
    (df_raw["processing_date_hour"] == target_processing_hour).sum()
)

print("Target processing hour, label = 1:")
print("-", target_processing_hour, "| rows:", target_count)

print("\nComparison processing hours, label = 0:")
for h in comparison_processing_hours:
    cnt = int((df_raw["processing_date_hour"] == h).sum())
    print("-", h, "| rows:", cnt)

Target processing hour, label = 1:
- 2025-01-15 12:00:00 | rows: 32738

Comparison processing hours, label = 0:
- 2025-01-15 00:00:00 | rows: 86467
- 2025-01-14 12:00:00 | rows: 30435
- 2025-01-08 12:00:00 | rows: 28758
- 2025-01-01 12:00:00 | rows: 36464


In [12]:
def sample_by_processing_hour(
    source_df,
    processing_hour,
    label,
    max_rows,
    random_state
):
    part = source_df[source_df["processing_date_hour"] == processing_hour].copy()

    if len(part) == 0:
        return part

    if len(part) > max_rows:
        part = part.sample(max_rows, random_state=random_state)

    part["label"] = label

    return part

In [13]:
positive_df = sample_by_processing_hour(
    source_df=df_raw,
    processing_hour=target_processing_hour,
    label=1,
    max_rows=TARGET_SAMPLE_PER_GROUP,
    random_state=RANDOM_STATE
)

print("Positive sample:")
print("Processing hour:", target_processing_hour)
print("Rows:", len(positive_df))

display(
    positive_df[[
        "date_hour",
        "processing_date_hour",
        "label"
    ]].head()
)

if len(positive_df) < MIN_ROWS_PER_GROUP:
    raise ValueError(
        f"Positive sample quá ít: {len(positive_df)} rows. "
        f"Cần tối thiểu {MIN_ROWS_PER_GROUP} rows."
    )

Positive sample:
Processing hour: 2025-01-15 12:00:00
Rows: 10000


,date_hour,processing_date_hour,label
1248923,2025-01-15 10:00:00,2025-01-15 12:00:00,1
1253056,2025-01-15 09:00:00,2025-01-15 12:00:00,1
1239600,2025-01-15 06:00:00,2025-01-15 12:00:00,1
1236295,2025-01-15 00:00:00,2025-01-15 12:00:00,1
1234755,2025-01-15 00:00:00,2025-01-15 12:00:00,1


In [14]:
negative_parts = []

for i, compare_hour in enumerate(comparison_processing_hours):
    part = sample_by_processing_hour(
        source_df=df_raw,
        processing_hour=compare_hour,
        label=0,
        max_rows=TARGET_SAMPLE_PER_GROUP,
        random_state=RANDOM_STATE + i + 100
    )

    print("Comparison hour:", compare_hour, "| rows:", len(part))

    if len(part) >= MIN_ROWS_PER_GROUP:
        negative_parts.append(part)
    else:
        print("Skipped because rows < MIN_ROWS_PER_GROUP")

if len(negative_parts) == 0:
    raise ValueError("Không có đủ dữ liệu đã có để tạo label = 0.")

negative_df = pd.concat(negative_parts, axis=0)

print("\nNegative sample total rows:", len(negative_df))

display(
    negative_df.groupby("processing_date_hour")
    .size()
    .reset_index(name="row_count")
    .sort_values("processing_date_hour")
)

Comparison hour: 2025-01-15 00:00:00 | rows: 10000
Comparison hour: 2025-01-14 12:00:00 | rows: 10000
Comparison hour: 2025-01-08 12:00:00 | rows: 10000
Comparison hour: 2025-01-01 12:00:00 | rows: 10000

Negative sample total rows: 40000


,processing_date_hour,row_count
0,2025-01-01 12:00:00,10000
1,2025-01-08 12:00:00,10000
2,2025-01-14 12:00:00,10000
3,2025-01-15 00:00:00,10000


In [15]:
work_df = pd.concat([positive_df, negative_df], axis=0)
work_df = work_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("Work dataset shape:", work_df.shape)

print("\nLabel count:")
print(work_df["label"].value_counts())

print("\nLabel ratio:")
print(work_df["label"].value_counts(normalize=True))

print("\nRows by label and source_processing_date_hour:")
display(
    work_df.groupby(["label", "processing_date_hour"])
    .size()
    .reset_index(name="row_count")
    .sort_values(["label", "processing_date_hour"])
)

Work dataset shape: (50000, 23)

Label count:
label
0    40000
1    10000
Name: count, dtype: int64

Label ratio:
label
0    0.8
1    0.2
Name: proportion, dtype: float64

Rows by label and source_processing_date_hour:


,label,processing_date_hour,row_count
0,0,2025-01-01 12:00:00,10000
1,0,2025-01-08 12:00:00,10000
2,0,2025-01-14 12:00:00,10000
3,0,2025-01-15 00:00:00,10000
4,1,2025-01-15 12:00:00,10000


In [16]:
work_df.head()

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,...,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,date_hour,processing_date_hour,label
0,1,2025-01-08 08:43:22,2025-01-08 09:00:22,NaN,1.90,99.0,<NA>,158,100,0,...,0.00,0.0,1.0,23.01,NaN,NaN,0.75,2025-01-08 09:00:00,2025-01-08 12:00:00,0
1,2,2025-01-15 08:48:11,2025-01-15 08:59:42,1.0,2.01,1.0,N,144,234,1,...,3.51,0.0,1.0,21.06,2.5,0.0,0.75,2025-01-15 09:00:00,2025-01-15 12:00:00,1
2,2,2025-01-15 09:12:03,2025-01-15 09:32:43,6.0,2.52,1.0,N,107,161,1,...,4.77,0.0,1.0,28.62,2.5,0.0,0.75,2025-01-15 09:00:00,2025-01-15 12:00:00,1
3,2,2025-01-14 15:06:00,2025-01-14 15:18:00,NaN,4.63,99.0,<NA>,78,208,0,...,0.00,0.0,1.0,19.64,NaN,NaN,0.00,2025-01-14 15:00:00,2025-01-15 00:00:00,0
4,2,2025-01-08 09:22:58,2025-01-08 09:42:08,1.0,1.96,1.0,N,90,163,1,...,4.49,0.0,1.0,26.94,2.5,0.0,0.75,2025-01-08 09:00:00,2025-01-08 12:00:00,0


In [17]:
EXCLUDE_COLS = [
    "label",
    "date_hour",
    "processing_date_hour",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
]

print("Columns to drop from model:")
for c in EXCLUDE_COLS:
    if c in work_df.columns:
        print("-", c)

Columns to drop from model:
- label
- date_hour
- processing_date_hour
- tpep_pickup_datetime
- tpep_dropoff_datetime


In [18]:
CATEGORICAL_COLS = [
    "VendorID",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
    "payment_type",
]

NUMERIC_COLS = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee",
]

CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in work_df.columns]
NUMERIC_COLS = [c for c in NUMERIC_COLS if c in work_df.columns]

print("Categorical columns:", CATEGORICAL_COLS)
print("Numeric columns:", NUMERIC_COLS)

Categorical columns: ['store_and_fwd_flag', 'payment_type']
Numeric columns: ['passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'cbd_congestion_fee']


## Encoding

In [19]:
class TaxiFeatureEncoder:
    def __init__(self, categorical_cols, numeric_cols, exclude_cols):
        self.categorical_cols = list(categorical_cols)
        self.numeric_cols = list(numeric_cols)
        self.exclude_cols = set(exclude_cols)

        self.category_maps_ = {}
        self.numeric_output_cols_ = []
        self.output_cols_ = []

    def _remove_excluded_columns(self, input_df):
        data = input_df.copy()

        cols_to_drop = [
            col for col in self.exclude_cols
            if col in data.columns
        ]

        if cols_to_drop:
            data = data.drop(columns=cols_to_drop)

        return data

    def fit(self, input_df):
        data = self._remove_excluded_columns(input_df)

        # Numeric: giữ các numeric_cols có trong data
        self.numeric_output_cols_ = [
            col for col in self.numeric_cols
            if col in data.columns
        ]

        self.category_maps_ = {}

        for col in self.categorical_cols:
            if col in data.columns:
                values = data[col].astype("string")

                # Chỉ lấy category từ giá trị không null
                categories = sorted(
                    values.dropna().unique().tolist()
                )

                self.category_maps_[col] = categories

        transformed = self.transform(input_df)
        self.output_cols_ = transformed.columns.tolist()

        return self

    def transform(self, input_df):
        data = self._remove_excluded_columns(input_df)
        out = pd.DataFrame(index=data.index)

        # Numeric: giữ NaN, không fill, không tạo __isnull
        for col in self.numeric_output_cols_:
            if col in data.columns:
                out[col] = pd.to_numeric(data[col], errors="coerce")

        # Categorical: giữ null là missing, không fill "__MISSING__"
        for col, categories in self.category_maps_.items():
            if col in data.columns:
                values = data[col].astype("string")

                out[col] = pd.Categorical(
                    values,
                    categories=categories
                )

        # Replace inf thành NaN để LightGBM xử lý
        out = out.replace([np.inf, -np.inf], np.nan)

        # Đảm bảo thứ tự cột giống lúc fit
        if self.output_cols_:
            for col in self.output_cols_:
                if col not in out.columns:
                    if col in self.category_maps_:
                        out[col] = pd.Categorical(
                            [pd.NA] * len(out),
                            categories=self.category_maps_[col]
                        )
                    else:
                        out[col] = np.nan

            out = out[self.output_cols_]

        return out

In [20]:
y = work_df["label"].astype(int)
X_raw = work_df.drop(columns=["label"], errors="ignore")
n_total = len(work_df)
n_test = int(round(TEST_SIZE * n_total))
n_valid = int(round(VALID_SIZE * n_total))

X_train_valid_raw, X_test_raw, y_train_valid, y_test = train_test_split(
    X_raw,
    y,
    test_size=n_test,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train_raw, X_valid_raw, y_train, y_valid = train_test_split(
    X_train_valid_raw,
    y_train_valid,
    test_size=n_valid,
    random_state=RANDOM_STATE,
    stratify=y_train_valid
)

print("Train raw:", X_train_raw.shape, "| positive rate:", y_train.mean())
print("Valid raw:", X_valid_raw.shape, "| positive rate:", y_valid.mean())
print("Test raw :", X_test_raw.shape, "| positive rate:", y_test.mean())

Train raw: (35000, 22) | positive rate: 0.2
Valid raw: (7500, 22) | positive rate: 0.2
Test raw : (7500, 22) | positive rate: 0.2


In [21]:
encoder = TaxiFeatureEncoder(
    categorical_cols=CATEGORICAL_COLS,
    numeric_cols=NUMERIC_COLS,
    exclude_cols=EXCLUDE_COLS
)

encoder.fit(X_train_raw)

X_train = encoder.transform(X_train_raw)
X_valid = encoder.transform(X_valid_raw)
X_test = encoder.transform(X_test_raw)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test :", X_test.shape)

print("Feature columns:")
print(X_train.columns.tolist())

X_train: (35000, 13)
X_valid: (7500, 13)
X_test : (7500, 13)
Feature columns:
['passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'cbd_congestion_fee', 'store_and_fwd_flag', 'payment_type']


In [22]:
missing_summary = (
    X_train
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(missing_summary[missing_summary > 0])

passenger_count         5459
congestion_surcharge    5459
store_and_fwd_flag      5459
dtype: int64

In [23]:
baseline_params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",

    # Số cây tối đa, early stopping sẽ tự chọn best_iteration
    "n_estimators": N_ESTIMATORS_MAX,

    # Baseline core params
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 20,

    # Baseline không dùng subsampling
    "subsample": 1.0,
    "subsample_freq": 0,
    "colsample_bytree": 1.0,

    # Regularization gần như tắt
    "reg_alpha": 1e-8,
    "reg_lambda": 1e-8,
    "min_split_gain": 0.0,

    # Binning mặc định
    "max_bin": 255,

    # Native missing handling
    "use_missing": True,
    "zero_as_missing": False,

    # Reproducibility / resource
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": -1,
}

baseline_model = lgb.LGBMClassifier(**baseline_params)

baseline_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

baseline_valid_pred = baseline_model.predict_proba(X_valid)[:, 1]
baseline_test_pred = baseline_model.predict_proba(X_test)[:, 1]

baseline_valid_auc = roc_auc_score(y_valid, baseline_valid_pred)
baseline_test_auc = roc_auc_score(y_test, baseline_test_pred)

print("Baseline valid AUC:", baseline_valid_auc)
print("Baseline test AUC :", baseline_test_auc)
print("Baseline best iteration:", baseline_model.best_iteration_)

Training until validation scores don't improve for 100 rounds
[50]	valid_0's auc: 0.690735
[100]	valid_0's auc: 0.690314
Early stopping, best iteration is:
[30]	valid_0's auc: 0.69221
Baseline valid AUC: 0.6922103888888889
Baseline test AUC : 0.6830474444444445
Baseline best iteration: 30


In [24]:
def suggest_lgbm_params(trial):
    # LightGBM leaf-wise tree:
    # Nếu max_depth được set, nên giữ num_leaves <= 2^max_depth
    max_depth = trial.suggest_categorical(
        "max_depth",
        [-1, 4, 5, 6, 7, 8, 10, 12]
    )

    if max_depth == -1:
        num_leaves_upper = 256
    else:
        num_leaves_upper = min(256, 2 ** max_depth)

    params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,

        # Native missing handling
        "use_missing": True,
        "zero_as_missing": False,

        # 1. Learning speed
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.15,
            log=True
        ),

        # 2. Tree complexity
        "num_leaves": trial.suggest_int(
            "num_leaves",
            16,
            num_leaves_upper
        ),

        "max_depth": max_depth,

        # 3. Prevent overfitting in leaf-wise trees
        # LightGBM default = 20, nên search bắt đầu từ 10 để không loại baseline
        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            10,
            1000,
            log=True
        ),

        # 4. Row bagging
        # subsample only works when subsample_freq > 0
        "subsample": trial.suggest_float(
            "subsample",
            0.7,
            1.0
        ),

        "subsample_freq": trial.suggest_int(
            "subsample_freq",
            0,
            10
        ),

        # 5. Feature sampling
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.7,
            1.0
        ),

        # 6. Regularization
        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            10.0,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-8,
            10.0,
            log=True
        ),

        # 7. Split control
        "min_split_gain": trial.suggest_float(
            "min_split_gain",
            0.0,
            1.0
        ),

        # 8. Histogram binning
        "max_bin": trial.suggest_categorical(
            "max_bin",
            [63, 127, 255]
        ),
    }

    return params

In [25]:
def objective(trial):
    params = suggest_lgbm_params(trial)

    model = lgb.LGBMClassifier(
        **params,
        n_estimators=N_ESTIMATORS_MAX
    )

    start_time = time.time()

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=EARLY_STOPPING_ROUNDS,
                verbose=False
            )
        ]
    )

    valid_pred = model.predict_proba(X_valid)[:, 1]
    valid_auc = roc_auc_score(y_valid, valid_pred)

    elapsed = time.time() - start_time

    trial.set_user_attr("valid_auc", float(valid_auc))
    trial.set_user_attr("best_iteration", int(model.best_iteration_))
    trial.set_user_attr("elapsed_seconds", float(elapsed))

    return valid_auc

In [26]:
sampler = optuna.samplers.TPESampler(
    seed=RANDOM_STATE,
    n_startup_trials=10
)

pruner = optuna.pruners.MedianPruner(
    n_startup_trials=5,
    n_warmup_steps=50,
    interval_steps=10
)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    pruner=pruner
)

print("Optuna study created.")

[I 2026-05-03 22:50:27,956] A new study created in memory with name: no-name-3e869c8c-cb9f-4762-bfa5-98617bafd97d


Optuna study created.


In [27]:
study.enqueue_trial({
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 20,
    "subsample": 1.0,
    "subsample_freq": 0,
    "colsample_bytree": 1.0,
    "reg_alpha": 1e-8,
    "reg_lambda": 1e-8,
    "min_split_gain": 0.0,
    "max_bin": 255,
})

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

print("Best validation AUC:", study.best_value)
print("Best params:")
print(study.best_params)

Best trial: 0. Best value: 0.69221:   1%|          | 1/100 [00:00<00:50,  1.98it/s]

[I 2026-05-03 22:50:30,605] Trial 0 finished with value: 0.6922103888888889 and parameters: {'max_depth': -1, 'learning_rate': 0.05, 'num_leaves': 31, 'min_child_samples': 20, 'subsample': 1.0, 'subsample_freq': 0, 'colsample_bytree': 1.0, 'reg_alpha': 1e-08, 'reg_lambda': 1e-08, 'min_split_gain': 0.0, 'max_bin': 255}. Best is trial 0 with value: 0.6922103888888889.


Best trial: 0. Best value: 0.69221:   2%|▏         | 2/100 [00:00<00:46,  2.12it/s]

[I 2026-05-03 22:50:31,055] Trial 1 finished with value: 0.6852301666666667 and parameters: {'max_depth': 4, 'learning_rate': 0.05092911283433821, 'num_leaves': 16, 'min_child_samples': 257, 'subsample': 0.7061753482887407, 'subsample_freq': 10, 'colsample_bytree': 0.9497327922401265, 'reg_alpha': 8.148018307012941e-07, 'reg_lambda': 4.329370014459266e-07, 'min_split_gain': 0.18340450985343382, 'max_bin': 127}. Best is trial 0 with value: 0.6922103888888889.


Best trial: 0. Best value: 0.69221:   3%|▎         | 3/100 [00:01<01:09,  1.39it/s]

...

[I 2026-05-03 22:52:19,061] Trial 97 finished with value: 0.6953241666666667 and parameters: {'max_depth': 10, 'learning_rate': 0.012118583748508096, 'num_leaves': 41, 'min_child_samples': 23, 'subsample': 0.9821873382386296, 'subsample_freq': 10, 'colsample_bytree': 0.9256934171827934, 'reg_alpha': 6.393963861742749e-08, 'reg_lambda': 5.895538715355359e-06, 'min_split_gain': 0.752147973755216, 'max_bin': 127}. Best is trial 51 with value: 0.6962298333333333.


Best trial: 51. Best value: 0.69623:  99%|█████████▉| 99/100 [01:50<00:02,  2.15s/it]

[I 2026-05-03 22:52:20,353] Trial 98 finished with value: 0.6940182777777778 and parameters: {'max_depth': 10, 'learning_rate': 0.011291441160486979, 'num_leaves': 44, 'min_child_samples': 30, 'subsample': 0.9775528987924498, 'subsample_freq': 10, 'colsample_bytree': 0.9203049816153436, 'reg_alpha': 4.7176503219201906e-08, 'reg_lambda': 7.946346509894711e-06, 'min_split_gain': 0.7463130681798622, 'max_bin': 127}. Best is trial 51 with value: 0.6962298333333333.


Best trial: 51. Best value: 0.69623: 100%|██████████| 100/100 [01:51<00:00,  1.12s/it]

[I 2026-05-03 22:52:21,875] Trial 99 finished with value: 0.6942389444444445 and parameters: {'max_depth': 10, 'learning_rate': 0.01272573582568805, 'num_leaves': 32, 'min_child_samples': 20, 'subsample': 0.9716433243361913, 'subsample_freq': 10, 'colsample_bytree': 0.9470671785824655, 'reg_alpha': 5.506290357144771e-08, 'reg_lambda': 1.3392273809089784e-05, 'min_split_gain': 0.6999204572952995, 'max_bin': 127}. Best is trial 51 with value: 0.6962298333333333.
Best validation AUC: 0.6962298333333333
Best params:
{'max_depth': 10, 'learning_rate': 0.051083723788855104, 'num_leaves': 81, 'min_child_samples': 15, 'subsample': 0.9813181710870581, 'subsample_freq': 8, 'colsample_bytree': 0.9129016411695934, 'reg_alpha': 0.05506488949139064, 'reg_lambda': 0.0016046179629749745, 'min_split_gain': 0.6541180611548333, 'max_bin': 127}


In [28]:
baseline_search_params = {
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 20,
    "subsample": 1.0,
    "subsample_freq": 0,
    "colsample_bytree": 1.0,
    "reg_alpha": 1e-8,
    "reg_lambda": 1e-8,
    "min_split_gain": 0.0,
    "max_bin": 255,
}

if study.best_value >= baseline_valid_auc:
    selected_params = study.best_params
    selected_param_source = "optuna"
    selected_valid_auc = study.best_value
else:
    selected_params = baseline_search_params
    selected_param_source = "baseline"
    selected_valid_auc = baseline_valid_auc

print("Selected param source:", selected_param_source)
print("Selected validation AUC:", selected_valid_auc)
print("Selected params:")
print(selected_params)

Selected param source: optuna
Selected validation AUC: 0.6962298333333333
Selected params:
{'max_depth': 10, 'learning_rate': 0.051083723788855104, 'num_leaves': 81, 'min_child_samples': 15, 'subsample': 0.9813181710870581, 'subsample_freq': 8, 'colsample_bytree': 0.9129016411695934, 'reg_alpha': 0.05506488949139064, 'reg_lambda': 0.0016046179629749745, 'min_split_gain': 0.6541180611548333, 'max_bin': 127}


In [29]:
final_model = lgb.LGBMClassifier(
    objective="binary",
    metric="auc",
    boosting_type="gbdt",
    verbosity=-1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    n_estimators=N_ESTIMATORS_MAX,
    use_missing=True,
    zero_as_missing=False,
    **selected_params
)

final_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

final_valid_pred = final_model.predict_proba(X_valid)[:, 1]
final_test_pred = final_model.predict_proba(X_test)[:, 1]

final_valid_auc = roc_auc_score(y_valid, final_valid_pred)
final_test_auc = roc_auc_score(y_test, final_test_pred)

print("Selected param source:", selected_param_source)
print("Final valid AUC:", final_valid_auc)
print("Final test AUC :", final_test_auc)
print("Final best iteration:", final_model.best_iteration_)

Training until validation scores don't improve for 100 rounds
[50]	valid_0's auc: 0.695392
[100]	valid_0's auc: 0.692439
Early stopping, best iteration is:
[33]	valid_0's auc: 0.69623
Selected param source: optuna
Final valid AUC: 0.6962298333333333
Final test AUC : 0.6840723333333334
Final best iteration: 33


In [30]:
auc_summary_df = pd.DataFrame([
    {
        "case": "baseline",
        "valid_auc": float(baseline_valid_auc),
        "test_auc": float(baseline_test_auc),
        "param_source": "baseline"
    },
    {
        "case": "optuna_best",
        "valid_auc": float(study.best_value),
        "test_auc": None,
        "param_source": "optuna"
    },
    {
        "case": "final_selected",
        "valid_auc": float(final_valid_auc),
        "test_auc": float(final_test_auc),
        "param_source": selected_param_source
    },
])

display(auc_summary_df)

,case,valid_auc,test_auc,param_source
0,baseline,0.69221,0.683047,baseline
1,optuna_best,0.69623,NaN,optuna
2,final_selected,0.69623,0.684072,optuna


In [31]:
trials_df = study.trials_dataframe()

display(
    trials_df.sort_values("value", ascending=False).head(10)
)

trials_path = f"{OUTPUT_DIR}/optuna_lgbm_trials.csv"
trials_df.to_csv(trials_path, index=False)

print("Saved:", trials_path)

,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_learning_rate,params_max_bin,params_max_depth,params_min_child_samples,...,params_num_leaves,params_reg_alpha,params_reg_lambda,params_subsample,params_subsample_freq,user_attrs_best_iteration,user_attrs_elapsed_seconds,user_attrs_valid_auc,system_attrs_fixed_params,state
51,51,0.696230,2026-05-03 22:51:28.789399,2026-05-03 22:51:29.370573,0 days 00:00:00.581174,0.912902,0.051084,127,10,15,...,81,5.506489e-02,1.604618e-03,0.981318,8,33,0.552788,0.696230,NaN,COMPLETE
94,94,0.695981,2026-05-03 22:52:09.327709,2026-05-03 22:52:10.456189,0 days 00:00:01.128480,0.919411,0.016343,127,10,16,...,43,8.564975e-05,5.887128e-06,0.979131,10,147,1.096291,0.695981,NaN,COMPLETE
96,96,0.695422,2026-05-03 22:52:12.029930,2026-05-03 22:52:13.783568,0 days 00:00:01.753638,0.924382,0.012031,127,10,15,...,42,2.632812e-08,5.216102e-06,0.978010,10,174,1.722038,0.695422,NaN,COMPLETE
97,97,0.695324,2026-05-03 22:52:13.789617,2026-05-03 22:52:19.061402,0 days 00:00:05.271785,0.925693,0.012119,127,10,23,...,41,6.393964e-08,5.895539e-06,0.982187,10,172,5.239044,0.695324,NaN,COMPLETE
60,60,0.695294,2026-05-03 22:51:35.727895,2026-05-03 22:51:36.571750,0 days 00:00:00.843855,0.891969,0.047292,127,10,12,...,61,1.919826e-04,9.445517e-06,0.937984,9,29,0.815329,0.695294,NaN,COMPLETE
95,95,0.695244,2026-05-03 22:52:10.461974,2026-05-03 22:52:12.024314,0 days 00:00:01.562340,0.919560,0.013773,127,10,16,...,43,4.077188e-08,6.575086e-06,0.980268,10,169,1.528755,0.695244,NaN,COMPLETE
25,25,0.695001,2026-05-03 22:51:01.204223,2026-05-03 22:51:02.306297,0 days 00:00:01.102074,0.880561,0.023873,127,12,13,...,41,9.826987e-05,7.732030e-03,0.971327,6,72,1.074792,0.695001,NaN,COMPLETE
64,64,0.694816,2026-05-03 22:51:39.107603,2026-05-03 22:51:44.029606,0 days 00:00:04.922003,0.856421,0.031396,127,10,11,...,72,1.673811e-04,2.801061e-07,0.976220,10,45,4.892325,0.694816,NaN,COMPLETE
70,70,0.694756,2026-05-03 22:51:48.642925,2026-05-03 22:51:49.353757,0 days 00:00:00.710832,0.845551,0.043590,127,10,16,...,61,2.147531e-05,1.592465e-06,0.940939,9,24,0.681539,0.694756,NaN,COMPLETE
62,62,0.694576,2026-05-03 22:51:37.384279,2026-05-03 22:51:38.275204,0 days 00:00:00.890925,0.858095,0.044528,127,10,12,...,61,1.884325e-04,1.429320e-06,0.957879,10,45,0.859759,0.694576,NaN,COMPLETE


Saved: /opt/spark/workspace/output/lgbm_experiment/optuna_lgbm_trials.csv


In [ ]:
from optuna.importance import get_param_importances

param_importance = get_param_importances(study)

param_importance_df = pd.DataFrame({
    "parameter": list(param_importance.keys()),
    "importance": list(param_importance.values())
}).sort_values("importance", ascending=False)

display(param_importance_df)

param_importance_path = f"{OUTPUT_DIR}/optuna_param_importance.csv"
param_importance_df.to_csv(param_importance_path, index=False)

print("Saved:", param_importance_path)

In [ ]:
best_config = {
    "model": "LightGBM",
    "task": "tlc_yellow_taxi_current_batch_vs_existing_batches",
    "target_processing_date_hour": str(TARGET_PROCESSING_DATE_HOUR),
    "comparison_processing_date_hours": [
        str(h) for h in comparison_processing_hours
    ],
    "metric": "auc",

    "baseline_valid_auc": float(baseline_valid_auc),
    "baseline_test_auc": float(baseline_test_auc),

    "optuna_best_valid_auc": float(study.best_value),
    "optuna_best_params": study.best_params,

    "selected_param_source": selected_param_source,
    "selected_valid_auc": float(selected_valid_auc),
    "selected_params": selected_params,

    "final_valid_auc": float(final_valid_auc),
    "final_test_auc": float(final_test_auc),
    "final_best_iteration": int(final_model.best_iteration_),

    "fixed_training_config": {
        "n_estimators_max": N_ESTIMATORS_MAX,
        "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
        "random_state": RANDOM_STATE,
        "target_sample_per_group": TARGET_SAMPLE_PER_GROUP,
        "min_rows_per_group": MIN_ROWS_PER_GROUP,
        "previous_batch_offset": str(PREVIOUS_BATCH_OFFSET),
        "history_offsets": [str(x) for x in HISTORY_OFFSETS],
        "use_missing": True,
        "zero_as_missing": False,
        "no_isnull_features": True,
        "no_trip_duration_min": True,
    },

    "tuned_parameter_groups": {
        "tree_complexity": [
            "num_leaves",
            "max_depth",
            "min_child_samples"
        ],
        "learning_process": [
            "learning_rate",
            "n_estimators_with_early_stopping"
        ],
        "sampling": [
            "subsample",
            "subsample_freq",
            "colsample_bytree"
        ],
        "regularization": [
            "reg_alpha",
            "reg_lambda",
            "min_split_gain"
        ],
        "binning": [
            "max_bin"
        ]
    }
}

best_params_path = f"{OUTPUT_DIR}/best_lgbm_params.json"

with open(best_params_path, "w") as f:
    json.dump(best_config, f, indent=2, default=str)

print("Saved:", best_params_path)
display(best_config)

In [ ]:
X_final_train = pd.concat([X_train, X_valid], axis=0)
y_final_train = pd.concat([y_train, y_valid], axis=0)

print("Final train:", X_final_train.shape)
print("Final test :", X_test.shape)

In [ ]:
threshold = 0.2
final_test_label_pred = (final_test_pred >= threshold).astype(int)

print(classification_report(y_test, final_test_label_pred))

In [ ]:
importance_df = pd.DataFrame({
    "feature": X_final_train.columns,
    "importance": final_model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance_df.head(30))

importance_path = f"{OUTPUT_DIR}/final_lgbm_feature_importance.csv"
importance_df.to_csv(importance_path, index=False)

print("Saved:", importance_path)

In [ ]:
import matplotlib.pyplot as plt

top_n = 20
plot_df = importance_df.head(top_n).sort_values("importance")

plt.figure(figsize=(8, 6))
plt.barh(plot_df["feature"], plot_df["importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top LightGBM Feature Importance")
plt.show()

In [ ]:
import shap

SHAP_SAMPLE_SIZE = min(3000, len(X_test))
# SHAP_SAMPLE_SIZE = len(X_test)
X_shap = X_test.sample(
    SHAP_SAMPLE_SIZE,
    random_state=RANDOM_STATE
)

explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_shap)

if isinstance(shap_values, list):
    shap_values_to_plot = shap_values[1]
else:
    shap_values_to_plot = shap_values

shap.summary_plot(
    shap_values_to_plot,
    X_shap,
    max_display=20
)

In [ ]:
shap.summary_plot(
    shap_values_to_plot,
    X_shap,
    plot_type="bar",
    max_display=20
)

In [ ]:
# Tỷ lệ bản ghi label = 1 bị giả lập bất thường
# 0.30 nghĩa là 30% bản ghi trong batch target sẽ bị chỉnh bất thường
ANOMALY_FRACTION = 0.30

# Seed để kết quả tái lập được
ANOMALY_RANDOM_STATE = 2025

print("Synthetic anomaly fraction:", ANOMALY_FRACTION)

In [ ]:
print("Positive target rows:", len(positive_df))
print("Negative history rows:", len(negative_df))

print("Target processing hour:")
print(positive_df["processing_date_hour"].value_counts())

print("History processing hours:")
display(
    negative_df["processing_date_hour"]
    .value_counts()
    .sort_index()
)

In [ ]:
def inject_synthetic_anomalies(
    target_df,
    anomaly_fraction=0.30,
    random_state=42
):
    """
    Giả lập bất thường trên dữ liệu target label = 1.
    
    Lưu ý:
    - Chỉ chỉnh dữ liệu target.
    - Không chỉnh dữ liệu lịch sử.
    - Cột synthetic_anomaly chỉ dùng để kiểm tra, không đưa vào model.
    """
    data = target_df.copy()
    rng = np.random.default_rng(random_state)

    data["synthetic_anomaly"] = 0

    n_rows = len(data)
    n_anomaly = int(n_rows * anomaly_fraction)

    if n_anomaly <= 0:
        return data

    anomaly_indices = rng.choice(
        data.index.to_numpy(),
        size=n_anomaly,
        replace=False
    )

    data.loc[anomaly_indices, "synthetic_anomaly"] = 1

    # Chia nhóm anomaly thành nhiều kiểu khác nhau
    anomaly_groups = np.array_split(anomaly_indices, 5)

    # 1. Fare / total amount tăng bất thường
    rows = anomaly_groups[0]
    if len(rows) > 0:
        if "fare_amount" in data.columns:
            multiplier = rng.uniform(3, 8, size=len(rows))
            data.loc[rows, "fare_amount"] = (
                pd.to_numeric(data.loc[rows, "fare_amount"], errors="coerce")
                .fillna(0)
                * multiplier
            )

        if "tip_amount" in data.columns:
            data.loc[rows, "tip_amount"] = rng.uniform(20, 100, size=len(rows))

        if "total_amount" in data.columns:
            base_fare = (
                pd.to_numeric(data.loc[rows, "fare_amount"], errors="coerce")
                .fillna(0)
            )
            data.loc[rows, "total_amount"] = base_fare + rng.uniform(50, 150, size=len(rows))

    # 2. Trip distance rất lớn
    rows = anomaly_groups[1]
    if len(rows) > 0 and "trip_distance" in data.columns:
        data.loc[rows, "trip_distance"] = rng.uniform(80, 180, size=len(rows))

        if "total_amount" in data.columns:
            data.loc[rows, "total_amount"] = rng.uniform(200, 500, size=len(rows))

    # 3. Passenger count bất thường hoặc missing
    rows = anomaly_groups[2]
    if len(rows) > 0 and "passenger_count" in data.columns:
        half = len(rows) // 2
        rows_high = rows[:half]
        rows_null = rows[half:]

        if len(rows_high) > 0:
            data.loc[rows_high, "passenger_count"] = rng.integers(8, 15, size=len(rows_high))

        if len(rows_null) > 0:
            data.loc[rows_null, "passenger_count"] = np.nan

    # 4. Category lạ trong payment_type / store_and_fwd_flag
    rows = anomaly_groups[3]
    if len(rows) > 0:
        if "payment_type" in data.columns:
            data.loc[rows, "payment_type"] = 99

        if "store_and_fwd_flag" in data.columns:
            data.loc[rows, "store_and_fwd_flag"] = "ANOM"

    # 5. Location ID bất thường
    rows = anomaly_groups[4]
    if len(rows) > 0:
        if "PULocationID" in data.columns:
            data.loc[rows, "PULocationID"] = 999

        if "DOLocationID" in data.columns:
            data.loc[rows, "DOLocationID"] = 999

    return data

In [ ]:
abnormal_positive_df = inject_synthetic_anomalies(
    target_df=positive_df,
    anomaly_fraction=ANOMALY_FRACTION,
    random_state=ANOMALY_RANDOM_STATE
)

print("Abnormal positive shape:", abnormal_positive_df.shape)
print("Synthetic anomaly counts:")
print(abnormal_positive_df["synthetic_anomaly"].value_counts())

display(
    abnormal_positive_df[
        [
            "processing_date_hour",
            "label",
            "synthetic_anomaly",
            "trip_distance",
            "fare_amount",
            "tip_amount",
            "total_amount",
            "passenger_count",
            "payment_type",
            "PULocationID",
            "DOLocationID",
        ]
    ].head(20)
)

In [ ]:
# Lịch sử giữ nguyên, thêm synthetic_anomaly = 0 để dễ phân tích
normal_negative_df = negative_df.copy()
normal_negative_df["synthetic_anomaly"] = 0

abnormal_work_df = pd.concat(
    [abnormal_positive_df, normal_negative_df],
    axis=0
)

abnormal_work_df = abnormal_work_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

print("Abnormal work dataset shape:", abnormal_work_df.shape)
print(abnormal_work_df["label"].value_counts())
print(abnormal_work_df["label"].value_counts(normalize=True))

display(
    abnormal_work_df.groupby(["label", "synthetic_anomaly"])
    .size()
    .reset_index(name="row_count")
)

In [ ]:
compare_cols = [
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "passenger_count",
]

compare_cols = [c for c in compare_cols if c in abnormal_work_df.columns]

normal_target_stats = positive_df[compare_cols].describe().T
abnormal_target_stats = abnormal_positive_df[compare_cols].describe().T
history_stats = negative_df[compare_cols].describe().T

summary_stats = pd.concat(
    {
        "normal_target": normal_target_stats[["mean", "50%", "max"]],
        "abnormal_target": abnormal_target_stats[["mean", "50%", "max"]],
        "history": history_stats[["mean", "50%", "max"]],
    },
    axis=1
)

display(summary_stats)

In [ ]:
y_abnormal = abnormal_work_df["label"].astype(int)
X_abnormal_raw = abnormal_work_df.drop(columns=["label", "synthetic_anomaly"], errors="ignore")

n_abn_total = len(abnormal_work_df)
n_abn_test = int(round(TEST_SIZE * n_abn_total))
n_anb_valid = int(round(VALID_SIZE * n_abn_total))

X_ab_train_valid_raw, X_ab_test_raw, y_ab_train_valid, y_ab_test = train_test_split(
    X_abnormal_raw,
    y_abnormal,
    test_size=n_abn_test,
    random_state=RANDOM_STATE,
    stratify=y_abnormal
)

X_ab_train_raw, X_ab_valid_raw, y_ab_train, y_ab_valid = train_test_split(
    X_ab_train_valid_raw,
    y_ab_train_valid,
    test_size=n_anb_valid,
    random_state=RANDOM_STATE,
    stratify=y_ab_train_valid
)

print("Abnormal train raw:", X_ab_train_raw.shape, "| positive rate:", y_ab_train.mean())
print("Abnormal valid raw:", X_ab_valid_raw.shape, "| positive rate:", y_ab_valid.mean())
print("Abnormal test raw :", X_ab_test_raw.shape, "| positive rate:", y_ab_test.mean())

In [ ]:
abnormal_encoder = TaxiFeatureEncoder(
    categorical_cols=CATEGORICAL_COLS,
    numeric_cols=NUMERIC_COLS,
    exclude_cols=EXCLUDE_COLS
)

abnormal_encoder.fit(X_ab_train_raw)

X_ab_train = abnormal_encoder.transform(X_ab_train_raw)
X_ab_valid = abnormal_encoder.transform(X_ab_valid_raw)
X_ab_test = abnormal_encoder.transform(X_ab_test_raw)

print("X_ab_train:", X_ab_train.shape)
print("X_ab_valid:", X_ab_valid.shape)
print("X_ab_test :", X_ab_test.shape)

print("Features used by model:")
print(X_ab_train.columns.tolist())

if "synthetic_anomaly" in X_ab_train.columns:
    raise ValueError("synthetic_anomaly bị đưa vào model. Cần loại bỏ ngay.")
else:
    print("OK: synthetic_anomaly is NOT used as a model feature.")

In [ ]:
abnormal_model = lgb.LGBMClassifier(
    objective="binary",
    metric="auc",
    boosting_type="gbdt",
    verbosity=-1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    n_estimators=N_ESTIMATORS_MAX,
    **study.best_params
)

abnormal_model.fit(
    X_ab_train,
    y_ab_train,
    eval_set=[(X_ab_valid, y_ab_valid)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

ab_valid_pred = abnormal_model.predict_proba(X_ab_valid)[:, 1]
ab_test_pred = abnormal_model.predict_proba(X_ab_test)[:, 1]

abnormal_valid_auc = roc_auc_score(y_ab_valid, ab_valid_pred)
abnormal_test_auc = roc_auc_score(y_ab_test, ab_test_pred)

print("Abnormal valid AUC:", abnormal_valid_auc)
print("Abnormal test AUC :", abnormal_test_auc)
print("Best iteration:", abnormal_model.best_iteration_)

In [ ]:
comparison_rows = []

# Các biến này được tạo ở phần notebook trước
# baseline_test_auc: baseline LightGBM bình thường
# final_test_auc: model sau Optuna trên dữ liệu bình thường

if "baseline_test_auc" in globals():
    comparison_rows.append({
        "case": "Normal data - baseline LightGBM",
        "test_auc": float(baseline_test_auc)
    })

if "final_test_auc" in globals():
    comparison_rows.append({
        "case": "Normal data - tuned LightGBM",
        "test_auc": float(final_test_auc)
    })

comparison_rows.append({
    "case": f"Synthetic anomaly data - tuned LightGBM, anomaly_fraction={ANOMALY_FRACTION}",
    "test_auc": float(abnormal_test_auc)
})

auc_comparison_df = pd.DataFrame(comparison_rows)

display(auc_comparison_df)

In [ ]:
score_analysis_df = X_ab_test_raw.copy()
score_analysis_df["true_label"] = y_ab_test.values
score_analysis_df["pred_score"] = ab_test_pred

if "synthetic_anomaly" not in score_analysis_df.columns:
    score_analysis_df["synthetic_anomaly"] = 0

display(
    score_analysis_df.groupby(["true_label", "synthetic_anomaly"])["pred_score"]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
)

In [ ]:
ab_importance_df = pd.DataFrame({
    "feature": X_ab_train.columns,
    "importance": abnormal_model.feature_importances_
}).sort_values("importance", ascending=False)

display(ab_importance_df.head(30))

ab_importance_path = f"{OUTPUT_DIR}/abnormal_lgbm_feature_importance.csv"
ab_importance_df.to_csv(ab_importance_path, index=False)

print("Saved:", ab_importance_path)

In [ ]:
import matplotlib.pyplot as plt

top_n = 20
plot_df = ab_importance_df.head(top_n).sort_values("importance")

plt.figure(figsize=(8, 6))
plt.barh(plot_df["feature"], plot_df["importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top LightGBM Feature Importance - Synthetic Anomaly Case")
plt.show()

In [ ]:
import shap

SHAP_SAMPLE_SIZE = len(X_ab_test)

X_ab_shap = X_ab_test.sample(
    SHAP_SAMPLE_SIZE,
    random_state=RANDOM_STATE
)

ab_explainer = shap.TreeExplainer(abnormal_model)
ab_shap_values = ab_explainer.shap_values(X_ab_shap)

if isinstance(ab_shap_values, list):
    ab_shap_values_to_plot = ab_shap_values[1]
else:
    ab_shap_values_to_plot = ab_shap_values

shap.summary_plot(
    ab_shap_values_to_plot,
    X_ab_shap,
    max_display=20
)

In [ ]:
shap.summary_plot(
    shap_values_to_plot,
    X_shap,
    plot_type="bar",
    max_display=20
)

In [ ]:
def run_synthetic_anomaly_experiment(
    anomaly_fraction,
    random_state=42
):
    # 1. Inject anomaly vào target
    temp_positive = inject_synthetic_anomalies(
        target_df=positive_df,
        anomaly_fraction=anomaly_fraction,
        random_state=random_state
    )

    # 2. History giữ nguyên
    temp_negative = negative_df.copy()
    temp_negative["synthetic_anomaly"] = 0

    # 3. Gộp dataset
    temp_work_df = pd.concat([temp_positive, temp_negative], axis=0)
    temp_work_df = temp_work_df.sample(
        frac=1,
        random_state=random_state
    ).reset_index(drop=True)

    # 4. Split
    y_temp = temp_work_df["label"].astype(int)
    X_temp_raw = temp_work_df.drop(columns=["label", "synthetic_anomaly"], errors="ignore")
    
    n_temp_total = len(temp_work_df)
    n_temp_test = int(round(TEST_SIZE * n_temp_total))
    n_temp_valid = int(round(VALID_SIZE * n_temp_total))
    
    X_temp_train_valid_raw, X_temp_test_raw, y_temp_train_valid, y_temp_test = train_test_split(
        X_temp_raw,
        y_temp,
        test_size=n_temp_test,
        random_state=random_state,
        stratify=y_temp
    )

    X_temp_train_raw, X_temp_valid_raw, y_temp_train, y_temp_valid = train_test_split(
        X_temp_train_valid_raw,
        y_temp_train_valid,
        test_size=n_temp_valid,
        random_state=random_state,
        stratify=y_temp_train_valid
    )

    # 5. Encode
    temp_encoder = TaxiFeatureEncoder(
        categorical_cols=CATEGORICAL_COLS,
        numeric_cols=NUMERIC_COLS,
        exclude_cols=EXCLUDE_COLS
    )

    temp_encoder.fit(X_temp_train_raw)

    X_temp_train = temp_encoder.transform(X_temp_train_raw)
    X_temp_valid = temp_encoder.transform(X_temp_valid_raw)
    X_temp_test = temp_encoder.transform(X_temp_test_raw)

    # 6. Train bằng best params, không tuning lại
    temp_model = lgb.LGBMClassifier(
        objective="binary",
        metric="auc",
        boosting_type="gbdt",
        verbosity=-1,
        random_state=random_state,
        n_jobs=-1,
        n_estimators=N_ESTIMATORS_MAX,
        **study.best_params
    )

    temp_model.fit(
        X_temp_train,
        y_temp_train,
        eval_set=[(X_temp_valid, y_temp_valid)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=EARLY_STOPPING_ROUNDS,
                verbose=False
            )
        ]
    )

    valid_pred = temp_model.predict_proba(X_temp_valid)[:, 1]
    test_pred = temp_model.predict_proba(X_temp_test)[:, 1]

    valid_auc = roc_auc_score(y_temp_valid, valid_pred)
    test_auc = roc_auc_score(y_temp_test, test_pred)

    result = {
        "anomaly_fraction": anomaly_fraction,
        "valid_auc": float(valid_auc),
        "test_auc": float(test_auc),
        "best_iteration": int(temp_model.best_iteration_),
        "n_rows": int(len(temp_work_df)),
        "n_positive": int((temp_work_df["label"] == 1).sum()),
        "n_negative": int((temp_work_df["label"] == 0).sum()),
        "n_synthetic_anomaly": int(temp_work_df["synthetic_anomaly"].sum()),
    }

    return result

In [ ]:
ANOMALY_LEVELS = [0.0, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90,1]

sweep_results = []

for frac in ANOMALY_LEVELS:
    print("Running anomaly_fraction =", frac)

    result = run_synthetic_anomaly_experiment(
        anomaly_fraction=frac,
        random_state=ANOMALY_RANDOM_STATE
    )

    sweep_results.append(result)

sweep_df = pd.DataFrame(sweep_results)

display(sweep_df)

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(
    sweep_df["anomaly_fraction"],
    sweep_df["test_auc"],
    marker="o"
)
plt.xlabel("Synthetic anomaly fraction")
plt.ylabel("Test AUC")
plt.title("AUC vs Synthetic Anomaly Fraction")
plt.grid(True)
plt.show()

In [ ]:
BATCH_START = pd.Timestamp("2025-01-16 12:00:00")
BATCH_END = pd.Timestamp("2025-01-31 12:00:00")

TARGET_BATCH_LIST = pd.date_range(
    start=BATCH_START,
    end=BATCH_END,
    freq="12h"
).tolist()

print("Number of target batches:", len(TARGET_BATCH_LIST))
print("First target batch:", TARGET_BATCH_LIST[0])
print("Last target batch :", TARGET_BATCH_LIST[-1])

TARGET_BATCH_LIST[:5], TARGET_BATCH_LIST[-5:]

In [ ]:
if "selected_params" in globals():
    MODEL_PARAMS = selected_params
    PARAM_SOURCE = "selected_params"
elif "study" in globals():
    MODEL_PARAMS = study.best_params
    PARAM_SOURCE = "study.best_params"
else:
    raise ValueError("Không tìm thấy selected_params hoặc study.best_params.")

print("Using params from:", PARAM_SOURCE)
print(MODEL_PARAMS)

In [ ]:
EXCLUDE_COLS = [
    "label",
    "date_hour",
    "processing_date_hour",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
]

print("Exclude columns:")
for col in EXCLUDE_COLS:
    print("-", col)

In [ ]:
def sample_by_processing_hour(
    source_df,
    processing_hour,
    label,
    max_rows,
    random_state
):
    part = source_df[source_df["processing_date_hour"] == processing_hour].copy()

    if len(part) == 0:
        return part

    if len(part) > max_rows:
        part = part.sample(max_rows, random_state=random_state)

    part["label"] = label
    # part["source_processing_date_hour"] = processing_hour

    return part

In [ ]:
def get_comparison_processing_hours(target_processing_hour):
    comparison_hours = [
        target_processing_hour - PREVIOUS_BATCH_OFFSET,
    ]

    comparison_hours += [
        target_processing_hour - offset
        for offset in HISTORY_OFFSETS
    ]

    # Loại trùng và sắp xếp
    comparison_hours = sorted(list(set(comparison_hours)))

    return comparison_hours

In [ ]:
def build_dataset_for_target_batch(
    source_df,
    target_processing_hour,
    min_rows_per_group=MIN_ROWS_PER_GROUP,
    max_rows_per_group=TARGET_SAMPLE_PER_GROUP,
    random_state=RANDOM_STATE
):
    # 1. Positive sample: target batch
    positive_df = sample_by_processing_hour(
        source_df=source_df,
        processing_hour=target_processing_hour,
        label=1,
        max_rows=max_rows_per_group,
        random_state=random_state
    )

    if len(positive_df) < min_rows_per_group:
        return None, {
            "status": "skipped",
            "reason": "positive_rows_too_few",
            "target_processing_hour": str(target_processing_hour),
            "positive_rows": len(positive_df),
        }

    # 2. Negative sample: all comparison batches
    comparison_hours = get_comparison_processing_hours(target_processing_hour)

    negative_parts = []
    negative_info = []

    for i, compare_hour in enumerate(comparison_hours):
        part = sample_by_processing_hour(
            source_df=source_df,
            processing_hour=compare_hour,
            label=0,
            max_rows=max_rows_per_group,
            random_state=random_state + i + 100
        )

        negative_info.append({
            "comparison_hour": str(compare_hour),
            "rows": len(part)
        })

        if len(part) >= min_rows_per_group:
            negative_parts.append(part)

    if len(negative_parts) == 0:
        return None, {
            "status": "skipped",
            "reason": "negative_rows_too_few",
            "target_processing_hour": str(target_processing_hour),
            "positive_rows": len(positive_df),
            "negative_info": negative_info,
        }

    negative_df = pd.concat(negative_parts, axis=0)

    # 3. Gộp positive + negative
    work_df = pd.concat([positive_df, negative_df], axis=0)
    work_df = work_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    meta = {
        "status": "ok",
        "target_processing_hour": str(target_processing_hour),
        "positive_rows": int((work_df["label"] == 1).sum()),
        "negative_rows": int((work_df["label"] == 0).sum()),
        "total_rows": int(len(work_df)),
        "comparison_hours": [str(h) for h in comparison_hours],
        "negative_info": negative_info,
    }

    return work_df, meta

In [ ]:
def train_evaluate_one_batch(
    batch_df,
    model_params,
    random_state=RANDOM_STATE
):
    y = batch_df["label"].astype(int)
    X_raw = batch_df.drop(columns=["label"], errors="ignore")

    n_temp_total = len(batch_df)
    n_temp_test = int(round(TEST_SIZE * n_temp_total))
    n_temp_valid = int(round(VALID_SIZE * n_temp_total))
    
    X_train_valid_raw, X_test_raw, y_train_valid, y_test = train_test_split(
        X_raw,
        y,
        test_size=n_temp_test,
        random_state=random_state,
        stratify=y
    )

    X_train_raw, X_valid_raw, y_train, y_valid = train_test_split(
        X_train_valid_raw,
        y_train_valid,
        test_size=n_temp_valid,
        random_state=random_state,
        stratify=y_train_valid
    )

    encoder = TaxiFeatureEncoder(
        categorical_cols=CATEGORICAL_COLS,
        numeric_cols=NUMERIC_COLS,
        exclude_cols=EXCLUDE_COLS
    )

    encoder.fit(X_train_raw)

    X_train = encoder.transform(X_train_raw)
    X_valid = encoder.transform(X_valid_raw)
    X_test = encoder.transform(X_test_raw)

    model = lgb.LGBMClassifier(
        objective="binary",
        metric="auc",
        boosting_type="gbdt",
        verbosity=-1,
        random_state=random_state,
        n_jobs=-1,
        n_estimators=N_ESTIMATORS_MAX,
        use_missing=True,
        zero_as_missing=False,
        **model_params
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=EARLY_STOPPING_ROUNDS,
                verbose=False
            )
        ]
    )

    valid_pred = model.predict_proba(X_valid)[:, 1]
    test_pred = model.predict_proba(X_test)[:, 1]

    valid_auc = roc_auc_score(y_valid, valid_pred)
    test_auc = roc_auc_score(y_test, test_pred)

    result = {
        "valid_auc": float(valid_auc),
        "test_auc": float(test_auc),
        "best_iteration": int(model.best_iteration_),
        "n_train": int(len(X_train)),
        "n_valid": int(len(X_valid)),
        "n_test": int(len(X_test)),
        "positive_rate_train": float(y_train.mean()),
        "positive_rate_valid": float(y_valid.mean()),
        "positive_rate_test": float(y_test.mean()),
    }

    artifacts = {
        "model": model,
        "encoder": encoder,
        "X_train": X_train,
        "X_valid": X_valid,
        "X_test": X_test,
        "X_test_raw": X_test_raw,
        "y_train": y_train,
        "y_valid": y_valid,
        "y_test": y_test,
        "test_pred": test_pred,
    }

    return result, artifacts

In [ ]:
def compute_lgbm_shap_top_features(
    model,
    X_data,
    target_processing_hour,
    top_n=10,
    sample_size=1000,
    random_state=RANDOM_STATE
):
    if len(X_data) > sample_size:
        X_shap = X_data.sample(sample_size, random_state=random_state)
    else:
        X_shap = X_data.copy()

    # LightGBM native SHAP contributions
    # Output shape = n_rows x (n_features + 1)
    # Cột cuối là expected value / bias term
    shap_contrib = model.booster_.predict(
        X_shap,
        pred_contrib=True
    )

    shap_values = shap_contrib[:, :-1]

    mean_abs_shap = np.abs(shap_values).mean(axis=0)

    shap_df = pd.DataFrame({
        "target_processing_date_hour": target_processing_hour,
        "feature": X_shap.columns,
        "mean_abs_shap": mean_abs_shap
    }).sort_values("mean_abs_shap", ascending=False)

    shap_df["rank"] = range(1, len(shap_df) + 1)

    return shap_df.head(top_n), shap_df

In [ ]:
batch_auc_results = []
batch_shap_top_results = []
batch_shap_all_results = []
batch_run_logs = []

for idx, target_hour in enumerate(TARGET_BATCH_LIST):
    print("=" * 100)
    print(f"[{idx + 1}/{len(TARGET_BATCH_LIST)}] Running target batch:", target_hour)

    batch_df, meta = build_dataset_for_target_batch(
        source_df=df_raw,
        target_processing_hour=target_hour,
        min_rows_per_group=MIN_ROWS_PER_GROUP,
        max_rows_per_group=TARGET_SAMPLE_PER_GROUP,
        random_state=RANDOM_STATE + idx
    )

    batch_run_logs.append(meta)

    if batch_df is None:
        print("Skipped:", meta["reason"])
        continue

    print("Rows:", meta["total_rows"])
    print("Positive rows:", meta["positive_rows"])
    print("Negative rows:", meta["negative_rows"])
    print("Comparison hours:", meta["comparison_hours"])

    try:
        eval_result, artifacts = train_evaluate_one_batch(
            batch_df=batch_df,
            model_params=MODEL_PARAMS,
            random_state=RANDOM_STATE + idx
        )

        result_row = {
            "target_processing_date_hour": target_hour,
            "target_processing_date_hour_str": str(target_hour),
            "valid_auc": eval_result["valid_auc"],
            "test_auc": eval_result["test_auc"],
            "best_iteration": eval_result["best_iteration"],
            "total_rows": meta["total_rows"],
            "positive_rows": meta["positive_rows"],
            "negative_rows": meta["negative_rows"],
            "n_train": eval_result["n_train"],
            "n_valid": eval_result["n_valid"],
            "n_test": eval_result["n_test"],
            "positive_rate_test": eval_result["positive_rate_test"],
            "comparison_hours": " | ".join(meta["comparison_hours"]),
        }

        batch_auc_results.append(result_row)

        print("Valid AUC:", eval_result["valid_auc"])
        print("Test AUC :", eval_result["test_auc"])
        print("Best iteration:", eval_result["best_iteration"])

        shap_top_df, shap_all_df = compute_lgbm_shap_top_features(
            model=artifacts["model"],
            X_data=artifacts["X_test"],
            target_processing_hour=str(target_hour),
            top_n=10,
            sample_size=1000,
            random_state=RANDOM_STATE + idx
        )

        batch_shap_top_results.append(shap_top_df)
        batch_shap_all_results.append(shap_all_df)

        print("Top SHAP features:")
        display(shap_top_df[["rank", "feature", "mean_abs_shap"]])

        # Dọn nhẹ bộ nhớ
        del artifacts
        gc.collect()

    except Exception as e:
        print("Failed target batch:", target_hour)
        print("Error:", repr(e))

        batch_run_logs.append({
            "status": "failed",
            "target_processing_hour": str(target_hour),
            "error": repr(e)
        })

In [ ]:
auc_series_df = pd.DataFrame(batch_auc_results)

auc_series_df = auc_series_df.sort_values("target_processing_date_hour").reset_index(drop=True)

display(auc_series_df)

auc_series_path = f"{OUTPUT_DIR}/batch_auc_series.csv"
auc_series_df.to_csv(auc_series_path, index=False)

print("Saved:", auc_series_path)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))

plt.plot(
    auc_series_df["target_processing_date_hour"],
    auc_series_df["test_auc"],
    marker="o",
    label="Test AUC"
)

plt.axhline(
    0.5,
    linestyle="--",
    label="AUC = 0.5"
)

plt.xlabel("Target processing_date_hour")
plt.ylabel("Test AUC")
plt.title("Batch-level Test AUC over Time")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Số điểm lịch sử tối thiểu trước khi bắt đầu đánh giá bất thường
MIN_HISTORY_POINTS = 20

# Số điểm gần nhất dùng làm lịch sử.
# Nếu đặt None: dùng toàn bộ lịch sử trước đó.
# Nếu đặt 12: chỉ dùng 12 batch gần nhất.
ROLLING_WINDOW = None

# Mức báo động thống kê.
# Đây không phải threshold AUC, mà là mức p-value.
ALPHA = 0.05

print("MIN_HISTORY_POINTS:", MIN_HISTORY_POINTS)
print("ROLLING_WINDOW:", ROLLING_WINDOW)
print("ALPHA:", ALPHA)

In [ ]:
def add_empirical_auc_anomaly_score(
    auc_df,
    auc_col="test_auc",
    time_col="target_processing_date_hour",
    min_history_points=6,
    rolling_window=None,
    alpha=0.05
):
    """
    Tính anomaly score cho chuỗi AUC bằng empirical p-value.

    Ý tưởng:
    - Với mỗi điểm AUC hiện tại, lấy các AUC trước đó làm lịch sử.
    - Tính p-value một phía phía trên:
        p = (số điểm lịch sử >= AUC hiện tại + 1) / (số điểm lịch sử + 1)
    - p càng nhỏ => AUC hiện tại càng hiếm/cao bất thường so với lịch sử.

    Không dùng threshold AUC cố định.
    """

    data = auc_df.copy()
    data = data.sort_values(time_col).reset_index(drop=True)

    p_values = []
    percentile_ranks = []
    history_means = []
    history_medians = []
    history_q95s = []
    history_sizes = []
    anomaly_flags = []
    anomaly_reasons = []

    for i, row in data.iterrows():
        current_auc = row[auc_col]

        # Chỉ dùng dữ liệu quá khứ
        history = data.loc[:i-1, auc_col].dropna()

        if rolling_window is not None:
            history = history.tail(rolling_window)

        history_size = len(history)
        history_sizes.append(history_size)

        if history_size < min_history_points or pd.isna(current_auc):
            p_values.append(np.nan)
            percentile_ranks.append(np.nan)
            history_means.append(np.nan)
            history_medians.append(np.nan)
            history_q95s.append(np.nan)
            anomaly_flags.append(False)
            anomaly_reasons.append("warmup_not_enough_history")
            continue

        # Empirical p-value phía trên
        # +1 smoothing để tránh p-value = 0 khi điểm hiện tại lớn hơn toàn bộ lịch sử
        p_value = (np.sum(history >= current_auc) + 1) / (history_size + 1)

        # Percentile rank: tỷ lệ lịch sử <= current_auc
        percentile_rank = (np.sum(history <= current_auc) + 1) / (history_size + 1)

        history_mean = history.mean()
        history_median = history.median()
        history_q95 = history.quantile(1 - alpha)

        is_anomaly = p_value <= alpha

        p_values.append(float(p_value))
        percentile_ranks.append(float(percentile_rank))
        history_means.append(float(history_mean))
        history_medians.append(float(history_median))
        history_q95s.append(float(history_q95))
        anomaly_flags.append(bool(is_anomaly))

        if is_anomaly:
            anomaly_reasons.append(
                f"auc_is_in_top_{int(alpha * 100)}pct_tail_of_history"
            )
        else:
            anomaly_reasons.append("normal_vs_history")

    data["history_size"] = history_sizes
    data["history_auc_mean"] = history_means
    data["history_auc_median"] = history_medians
    data[f"history_auc_q{int((1-alpha)*100)}"] = history_q95s
    data["auc_empirical_p_value"] = p_values
    data["auc_percentile_rank"] = percentile_ranks
    data["is_auc_anomaly"] = anomaly_flags
    data["auc_anomaly_reason"] = anomaly_reasons

    return data

In [ ]:
auc_monitoring_df = add_empirical_auc_anomaly_score(
    auc_df=auc_series_df,
    auc_col="test_auc",
    time_col="target_processing_date_hour",
    min_history_points=MIN_HISTORY_POINTS,
    rolling_window=ROLLING_WINDOW,
    alpha=ALPHA
)

display(auc_monitoring_df)

In [ ]:
auc_anomaly_batches_df = auc_monitoring_df[
    auc_monitoring_df["is_auc_anomaly"] == True
].copy()

print("Number of AUC anomaly batches:", len(auc_anomaly_batches_df))

display(
    auc_anomaly_batches_df[
        [
            "target_processing_date_hour",
            "test_auc",
            "history_size",
            "history_auc_mean",
            "history_auc_median",
            f"history_auc_q{int((1-ALPHA)*100)}",
            "auc_empirical_p_value",
            "auc_percentile_rank",
            "auc_anomaly_reason",
        ]
    ]
)

In [ ]:
import matplotlib.pyplot as plt

q_col = f"history_auc_q{int((1-ALPHA)*100)}"

plt.figure(figsize=(14, 5))

plt.plot(
    auc_monitoring_df["target_processing_date_hour"],
    auc_monitoring_df["test_auc"],
    marker="o",
    label="Test AUC"
)

plt.plot(
    auc_monitoring_df["target_processing_date_hour"],
    auc_monitoring_df[q_col],
    marker="x",
    linestyle="--",
    label=f"Dynamic historical q{int((1-ALPHA)*100)}"
)

# Đánh dấu anomaly
anomaly_points = auc_monitoring_df[auc_monitoring_df["is_auc_anomaly"] == True]

plt.scatter(
    anomaly_points["target_processing_date_hour"],
    anomaly_points["test_auc"],
    s=120,
    marker="X",
    label="AUC anomaly"
)

plt.axhline(
    0.5,
    linestyle=":",
    label="AUC = 0.5"
)

plt.xlabel("Target processing_date_hour")
plt.ylabel("Test AUC")
plt.title("Dynamic AUC Monitoring using Empirical Historical Distribution")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    auc_monitoring_df["target_processing_date_hour"],
    auc_monitoring_df["auc_empirical_p_value"],
    marker="o",
    label="Empirical p-value"
)

plt.axhline(
    ALPHA,
    linestyle="--",
    label=f"alpha = {ALPHA}"
)

plt.xlabel("Target processing_date_hour")
plt.ylabel("Empirical p-value")
plt.title("AUC Empirical p-value over Time")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()